# run experiment

runs one experiment config on Colab. the code comes from GitHub at the chosen branch, results and figures go to the project folder on Drive under runs.

the GitHub repository and access token are read from Colab Secrets, GITHUB_REPO and GITHUB_TOKEN, so nothing personal is stored in this notebook.

In [ ]:
CONFIG = "configs/experiments/e00_smoke.yaml"
OVERRIDES = []  # for example ["seed=1", "smoke.epsilon=0.005"]
BRANCH = "main"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
from google.colab import userdata

CODE_DIR = "/content/forge-nids"
repo = userdata.get("GITHUB_REPO")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
auth = f"x-access-token:{token}@" if token else ""
url = f"https://{auth}github.com/{repo}.git"


def git(*args, cwd=None):
    out = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stderr.replace(token, "***") if token else out.stderr)
    return out.stdout.strip()


if os.path.isdir(CODE_DIR):
    # the Colab copy always matches GitHub exactly, local edits here are discarded
    git("fetch", "--quiet", url, BRANCH, cwd=CODE_DIR)
    git("checkout", "--quiet", "-B", BRANCH, "FETCH_HEAD", cwd=CODE_DIR)
else:
    git("clone", "--quiet", "--branch", BRANCH, url, CODE_DIR)
    git("remote", "set-url", "origin", f"https://github.com/{repo}.git", cwd=CODE_DIR)
print("code at", git("log", "-1", "--format=%h %s", cwd=CODE_DIR))

In [ ]:
%pip install --quiet -e /content/forge-nids

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/forge-nids")
os.environ["FORGE_STORAGE"] = str(DRIVE_ROOT)
for sub in ["notebooks", "data/raw", "data/processed", "runs", "exports"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no gpu" 

In [ ]:
import shlex

extra = "--set " + " ".join(shlex.quote(o) for o in OVERRIDES) if OVERRIDES else ""
!cd {CODE_DIR} && python scripts/run.py {CONFIG} {extra}

In [ ]:
import yaml
from IPython.display import Image, display

name = yaml.safe_load(open(f"{CODE_DIR}/{CONFIG}"))["experiment"]["name"]
latest = sorted((DRIVE_ROOT / "runs" / name).iterdir())[-1]
print(latest)
for png in sorted((latest / "figures").glob("*.png")):
    print(png.name)
    display(Image(filename=str(png), width=520))